# 03_feature_selection_and_train_test_split.ipynb

This notebook selects features, performs an 80/20 train-test split, applies StandardScaler, and saves the fitted scaler to `models/scaler.pkl`.
Ensure `data/processed/cleaned_data.csv` exists before running.

In [1]:
# Imports
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Paths
clean_path = os.path.join('data', 'processed', 'cleaned_data.csv')
scaler_path = os.path.join('models', 'scaler.pkl')
os.makedirs(os.path.dirname(scaler_path), exist_ok=True)

# Load cleaned dataset
if not os.path.exists(clean_path):
    raise FileNotFoundError(f'Cleaned dataset not found at {os.path.abspath(clean_path)}. Run preprocessing first.')
df = pd.read_csv(clean_path)
print('Loaded cleaned dataset with shape:', df.shape)

Loaded cleaned dataset with shape: (66647, 16)


In [2]:
# Display all columns
print('Columns:', list(df.columns))

Columns: ['age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio', 'age_years', 'height_m', 'BMI', 'pulse_pressure']


## Feature selection
We select the features listed in the project specification and set `cardio` as the target.

In [3]:
# Define features and target
features = [
    'age_years',
    'gender',
    'height',
    'weight',
    'BMI',
    'ap_hi',
    'ap_lo',
    'pulse_pressure',
    'cholesterol',
    'gluc',
    'smoke',
    'alco',
    'active'
]
target = 'cardio'

# Verify required columns exist
missing = [c for c in features + [target] if c not in df.columns]
if missing:
    raise KeyError(f'Missing required columns in cleaned data: {missing}')

# Create X and y
X = df[features].copy()  # input features
y = df[target].copy()    # target variable
print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (66647, 13)
y shape: (66647,)


## Train-test split
We split data into 80% training and 20% testing with `random_state=42` for reproducibility.

In [4]:
# Perform stratified train-test split to maintain class distribution
# stratify=y ensures both train and test sets have similar class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,  # 80/20 split
    random_state=42,  # for reproducibility
    stratify=y  # maintain class distribution in train/test sets
)
print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)
print(f'Class distribution in y_train: {y_train.value_counts().to_dict()}')
print(f'Class distribution in y_test: {y_test.value_counts().to_dict()}')


X_train shape: (53317, 13)
X_test shape: (13330, 13)
y_train shape: (53317,)
y_test shape: (13330,)
Class distribution in y_train: {0: 27250, 1: 26067}
Class distribution in y_test: {0: 6813, 1: 6517}


## Feature scaling
We apply `StandardScaler` so features have zero mean and unit variance which helps many ML algorithms converge faster.

In [5]:
# Fit scaler on training data and transform both train and test
scaler = StandardScaler()
scaler.fit(X_train)  # learn mean and scale from training data only
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print('X_train_scaled shape:', X_train_scaled.shape)
print('X_test_scaled shape:', X_test_scaled.shape)

# Save scaler for later use in model serving/prediction
joblib.dump(scaler, scaler_path)
print(f'Saved scaler to: {scaler_path}')

X_train_scaled shape: (53317, 13)
X_test_scaled shape: (13330, 13)
Saved scaler to: models\scaler.pkl


## Explanations
- **Why feature selection?** Removes irrelevant/noisy features, reduces overfitting, and improves training speed.
- **Why train-test split?** To estimate model generalization on unseen data and avoid optimistic performance estimates.
- **Why feature scaling?** Many models assume features are on similar scales; scaling avoids features with large ranges dominating and improves optimizer behavior.